# DINet Inference — Facial Dubbing

**What this does:** Given a face video and a driving audio file, DINet synthesizes a new video where the person's lips are synced to the audio.

## Before you start
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier)
2. Run all cells top to bottom
3. You will need to upload:
   - Your **source face video** (`.mp4`, 25fps recommended)
   - Your **driving audio** (`.wav`)
   - The **pretrained DINet model** (`clip_training_DINet_256mouth.pth`)
   - The **DeepSpeech model** (`output_graph.pb`)

> **Download pretrained models from the original DINet repo releases or Google Drive links provided by the authors.**

## Step 1 — Check GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → GPU (T4)')
print(f'GPU ready: {torch.cuda.get_device_name(0)}')

## Step 2 — Clone repo and install dependencies

In [ ]:
import os

if not os.path.exists('DINetDigital'):
    !git clone https://github.com/Sunil123135/DINetDigital.git

os.chdir('DINetDigital')
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q numpy==1.26.4 opencv-python-headless==4.9.0.80 python_speech_features resampy scipy
!pip install -q torch==1.13.1+cu116 torchvision==0.14.1+cu116 --extra-index-url https://download.pytorch.org/whl/cu116
!apt-get install -q ffmpeg
print('Dependencies installed.')

## Step 3 — Install OpenFace (for facial landmark extraction)
OpenFace is required to generate the `.csv` landmark file from your source video.

In [ ]:
# Install OpenFace pre-built binary
!wget -q https://github.com/TadasBaltrusaitis/OpenFace/releases/download/OpenFace_2.2.0/OpenFace_2.2.0_win_x64.zip 2>/dev/null || true

# For Linux (Colab), build from source or use the Docker approach
openface_dir = '/content/openface'
if not os.path.exists(openface_dir):
    !bash <(wget -q -O - https://raw.githubusercontent.com/TadasBaltrusaitis/OpenFace/master/download_models.sh) 2>/dev/null || true
    !sudo apt-get install -q libopenblas-dev liblapack-dev
    !git clone -q https://github.com/TadasBaltrusaitis/OpenFace.git /content/openface
    !cd /content/openface && bash download_models.sh && mkdir -p build && cd build && cmake -D CMAKE_BUILD_TYPE=RELEASE .. -DCMAKE_CXX_FLAGS="-march=native" > /dev/null && make -j4 2>&1 | tail -5
    print('OpenFace built.')
else:
    print('OpenFace already installed.')

## Step 4 — Upload your files
Upload:
- `source_video.mp4` — face video (25fps, person facing camera)
- `driving_audio.wav` — audio to dub onto the video
- `clip_training_DINet_256mouth.pth` — pretrained DINet model
- `output_graph.pb` — pretrained DeepSpeech model

In [ ]:
from google.colab import files
import shutil

os.makedirs('./asserts/examples', exist_ok=True)
os.makedirs('./asserts/inference_result', exist_ok=True)

print('Upload your source face video (.mp4):')
uploaded = files.upload()
source_video_name = list(uploaded.keys())[0]
shutil.move(source_video_name, './asserts/examples/source_video.mp4')
print(f'Saved source video: {source_video_name}')

In [ ]:
print('Upload your driving audio (.wav):')
uploaded = files.upload()
audio_name = list(uploaded.keys())[0]
shutil.move(audio_name, './asserts/examples/driving_audio.wav')
print(f'Saved audio: {audio_name}')

In [ ]:
print('Upload pretrained DINet model (.pth):')
uploaded = files.upload()
model_name = list(uploaded.keys())[0]
shutil.move(model_name, './asserts/clip_training_DINet_256mouth.pth')
print(f'Saved model: {model_name}')

In [ ]:
print('Upload DeepSpeech model (output_graph.pb):')
uploaded = files.upload()
ds_name = list(uploaded.keys())[0]
shutil.move(ds_name, './asserts/output_graph.pb')
print(f'Saved DeepSpeech model: {ds_name}')

## Step 5 — Convert video to 25fps (if needed)

In [ ]:
import cv2

cap = cv2.VideoCapture('./asserts/examples/source_video.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f'Source video FPS: {fps}')

if abs(fps - 25) > 0.5:
    print('Converting to 25fps...')
    !ffmpeg -i ./asserts/examples/source_video.mp4 -r 25 ./asserts/examples/source_video_25fps.mp4 -y -loglevel error
    shutil.move('./asserts/examples/source_video_25fps.mp4', './asserts/examples/source_video.mp4')
    print('Converted to 25fps.')
else:
    print('Video is already 25fps, no conversion needed.')

## Step 6 — Extract facial landmarks with OpenFace

In [ ]:
openface_bin = '/content/openface/build/bin/FeatureExtraction'

if not os.path.exists(openface_bin):
    raise RuntimeError('OpenFace build failed in Step 3. Please re-run Step 3.')

!{openface_bin} -f ./asserts/examples/source_video.mp4 -out_dir ./asserts/examples/ -2Dfp -loglevel error

# OpenFace names the output after the input file
import glob
csvs = glob.glob('./asserts/examples/*.csv')
if csvs:
    landmark_csv = csvs[0]
    shutil.copy(landmark_csv, './asserts/examples/source_video.csv')
    print(f'Landmarks extracted: {landmark_csv}')
else:
    raise RuntimeError('No landmark CSV found. OpenFace may have failed — check video quality/face visibility.')

## Step 7 — Run DINet Inference

In [ ]:
!python inference.py \
    --source_video_path ./asserts/examples/source_video.mp4 \
    --source_openface_landmark_path ./asserts/examples/source_video.csv \
    --driving_audio_path ./asserts/examples/driving_audio.wav \
    --pretrained_clip_DINet_path ./asserts/clip_training_DINet_256mouth.pth \
    --deepspeech_model_path ./asserts/output_graph.pb \
    --res_video_dir ./asserts/inference_result

## Step 8 — Preview and download result

In [ ]:
result_files = glob.glob('./asserts/inference_result/*.mp4')
print('Output files:')
for f in result_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

In [ ]:
# Preview the dubbed video in notebook
from IPython.display import HTML
from base64 import b64encode

# Pick the final audio+video merged file
final_files = [f for f in result_files if 'add_audio' in f]
if not final_files:
    final_files = result_files

if final_files:
    video_path = final_files[0]
    video_data = open(video_path, 'rb').read()
    video_b64 = b64encode(video_data).decode()
    display(HTML(f'''
        <video width="640" controls>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
    '''))
else:
    print('No output video found. Check Step 7 for errors.')

In [ ]:
# Download all result files
for f in result_files:
    print(f'Downloading {f}...')
    files.download(f)